# Statistika Deskriptif Metrik Kompleksitas Kode (LOC dan VG)
**Studi kasus: Software Defect Dataset by NASA (JM1)** · Statistika dan Probabilitas · 3 TI F · Kelompok 5

Alur mengikuti Lembar Perkuliahan Pertemuan 4 dan hanya memakai Pandas:

- **A. Pengorganisasian data**: import data dan tabel frekuensi (crosstab)
- **B. Penyajian data**: grafik garis, batang, pie, histogram, dan boxplot
- **C. Statistika deskriptif**: describe(), standard error, median, modus, varians, range, skewness, kurtosis

Letakkan file `3TIF_05_SoftwareDefect.xlsx` satu folder dengan notebook ini (di Colab: upload ke panel Files).

In [ ]:
import json
import os
import pandas as pd

FILE_XLSX = "3TIF_05_SoftwareDefect.xlsx"
os.makedirs("output/tabel", exist_ok=True)
os.makedirs("output/gambar", exist_ok=True)
print("Library dan pengaturan berhasil dimuat!")

## A. Pengorganisasian Data
Import data Excel, pemeriksaan data, dan pembuatan tabel frekuensi.

In [ ]:
dataraw = pd.read_excel(FILE_XLSX, sheet_name="Dataset", na_values=["NA"])
print("Ukuran data awal:", dataraw.shape)
print(dataraw.head(10))
print("\nMissing value per kolom:\n", dataraw.isna().sum())

# Baris dengan nilai desimal pada LOC/VG/EVG/IVG dianggap anomali dan dikeluarkan
anomali = dataraw[(dataraw[["LOC", "VG", "EVG", "IVG"]] % 1 != 0).any(axis=1)]
print("\nBaris anomali yang dikeluarkan:", len(anomali))
print(anomali)
df = dataraw.drop(anomali.index).reset_index(drop=True)
print("Jumlah observasi yang dianalisis:", len(df))

In [ ]:
# Variabel pengelompokan dalam bentuk label
df["Status"] = df["Cacat"].map({0: "Tidak cacat", 1: "Cacat"})

# Kategori ukuran (LOC) dan kompleksitas (VG) untuk tabel frekuensi
df["Kategori_LOC"] = pd.cut(df["LOC"], bins=[0, 10, 20, 50, 100, float("inf")],
                            labels=["1-10", "11-20", "21-50", "51-100", ">100"])
df["Kategori_VG"] = pd.cut(df["VG"], bins=[0, 10, 20, 50, float("inf")],
                           labels=["1-10", "11-20", "21-50", ">50"])


def tabel_frekuensi(kolom, urut=None):
    """Tabel frekuensi dengan crosstab (seperti pada lembar perkuliahan) + persentase."""
    frq = pd.crosstab(index=df[kolom], columns="Frekuensi")
    if urut is not None:
        frq = frq.reindex(urut)
    frq["Persentase"] = frq["Frekuensi"] / frq["Frekuensi"].sum() * 100
    return frq


frq_status = tabel_frekuensi("Status", ["Tidak cacat", "Cacat"])
frq_vg = tabel_frekuensi("Kategori_VG")
frq_loc = tabel_frekuensi("Kategori_LOC")
print("\nTabel frekuensi status cacat:\n", frq_status.round(2))
print("\nTabel frekuensi kategori VG:\n", frq_vg.round(2))
print("\nTabel frekuensi kategori LOC:\n", frq_loc.round(2))

# Crosstab kategori x status (jumlah dan persentase baris)
ct_vg = pd.crosstab(df["Kategori_VG"], df["Status"], margins=True)
ct_vg_pct = pd.crosstab(df["Kategori_VG"], df["Status"], normalize="index") * 100
ct_loc = pd.crosstab(df["Kategori_LOC"], df["Status"], margins=True)
ct_loc_pct = pd.crosstab(df["Kategori_LOC"], df["Status"], normalize="index") * 100
print("\nCrosstab VG x status:\n", ct_vg, "\n", ct_vg_pct.round(2))
print("\nCrosstab LOC x status:\n", ct_loc, "\n", ct_loc_pct.round(2))

for nama, t in [("frek_status", frq_status), ("frek_vg", frq_vg), ("frek_loc", frq_loc),
                ("crosstab_vg", ct_vg), ("crosstab_vg_pct", ct_vg_pct),
                ("crosstab_loc", ct_loc), ("crosstab_loc_pct", ct_loc_pct)]:
    t.to_csv(f"output/tabel/{nama}.csv")

## B. Penyajian Data
Data pada tabel frekuensi disajikan dalam grafik garis, batang, dan pie. Data numerik disajikan dengan histogram dan boxplot.

In [ ]:
import matplotlib.pyplot as plt  # hanya untuk mengatur tata letak dan menyimpan grafik Pandas

In [ ]:
def tiga_grafik(frq, judul, nama_file):
    """Grafik garis, batang, dan pie dari satu tabel frekuensi."""
    fig, ax = plt.subplots(1, 3, figsize=(13, 4))
    frq[["Frekuensi"]].plot(ax=ax[0], title="Grafik garis")
    frq[["Frekuensi"]].plot(kind="bar", ax=ax[1], title="Grafik batang")
    frq.plot(kind="pie", y="Frekuensi", autopct="%1.1f%%", legend=False, ax=ax[2], title="Pie chart")
    ax[2].set_ylabel("")
    fig.suptitle(judul)
    plt.tight_layout()
    plt.savefig(f"output/gambar/{nama_file}.png", bbox_inches="tight")
    plt.show()


# Status cacat: batang dan pie
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
frq_status[["Frekuensi"]].plot(kind="bar", ax=ax[0], title="Grafik batang")
frq_status.plot(kind="pie", y="Frekuensi", autopct="%1.1f%%", legend=False, ax=ax[1], title="Pie chart")
ax[1].set_ylabel("")
fig.suptitle("Distribusi frekuensi modul menurut status cacat")
plt.tight_layout()
plt.savefig("output/gambar/01_status_cacat.png", bbox_inches="tight")
plt.show()

tiga_grafik(frq_vg, "Distribusi frekuensi modul menurut kategori VG", "02_kategori_vg")
tiga_grafik(frq_loc, "Distribusi frekuensi modul menurut kategori LOC", "03_kategori_loc")

In [ ]:
# Persentase modul cacat pada tiap kategori
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ct_vg_pct["Cacat"].plot(kind="bar", ax=ax[0], color="#C44E52", title="Persentase modul cacat per kategori VG")
ct_loc_pct["Cacat"].plot(kind="bar", ax=ax[1], color="#C44E52", title="Persentase modul cacat per kategori LOC")
for a in ax:
    a.set_ylabel("Persentase modul cacat (%)")
plt.tight_layout()
plt.savefig("output/gambar/04_persen_cacat_per_kategori.png", bbox_inches="tight")
plt.show()

In [ ]:
# Histogram
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
df["LOC"].plot(kind="hist", bins=60, ax=ax[0], title="Histogram LOC")
df["VG"].plot(kind="hist", bins=60, ax=ax[1], title="Histogram VG")
ax[0].set_xlabel("LOC"); ax[1].set_xlabel("VG")
plt.tight_layout()
plt.savefig("output/gambar/05_histogram.png", bbox_inches="tight")
plt.show()

# Boxplot menurut status cacat (skala logaritma agar kedua kelompok terbaca)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
for a, v in zip(ax, ["LOC", "VG"]):
    df.boxplot(column=v, by="Status", ax=a)
    a.set_yscale("log")
    a.set_title(f"Boxplot {v} menurut status cacat (skala log)")
    a.set_xlabel("")
fig.suptitle("")
plt.tight_layout()
plt.savefig("output/gambar/06_boxplot.png", bbox_inches="tight")
plt.show()

## C. Statistika Deskriptif
Statistik dihitung untuk seluruh modul dan menurut status cacat.

In [ ]:
def statistik(dt):
    """Statistik deskriptif seperti pada lembar perkuliahan, ditambah IQR, KV, dan outlier."""
    stats = dt.describe()
    stats["Standard Error"] = dt.sem()
    stats["Median"] = dt.median()
    stats["Mode"] = dt.mode().iloc[0]
    stats["variance"] = dt.var()
    stats["range"] = dt.max() - dt.min()
    stats["skewness"] = dt.skew()
    stats["kurtosis"] = dt.kurtosis()
    q1, q3 = dt.quantile(0.25), dt.quantile(0.75)
    iqr = q3 - q1
    stats["IQR"] = iqr
    stats["KV (%)"] = dt.std() / dt.mean() * 100
    stats["Batas bawah"] = q1 - 1.5 * iqr
    stats["Batas atas"] = q3 + 1.5 * iqr
    stats["Jumlah outlier"] = int(((dt < q1 - 1.5 * iqr) | (dt > q3 + 1.5 * iqr)).sum())
    return stats


hasil = {}
for v in ["LOC", "VG"]:
    hasil[f"{v} | Seluruh modul"] = statistik(df[v])
    hasil[f"{v} | Tidak cacat"] = statistik(df.loc[df["Status"] == "Tidak cacat", v])
    hasil[f"{v} | Cacat"] = statistik(df.loc[df["Status"] == "Cacat", v])
tabel_stat = pd.DataFrame(hasil)
print("\nStatistika deskriptif:\n", tabel_stat.round(3).to_string())
tabel_stat.round(4).to_csv("output/tabel/statistika_deskriptif.csv")

# Simpan ringkasan angka untuk laporan
ringkas = {
    "n_awal": int(len(dataraw)), "n_analisis": int(len(df)),
    "n_missing_branchcount": int(dataraw["BranchCount"].isna().sum()),
    "frek_status": frq_status.to_dict(),
}
json.dump(ringkas, open("output/tabel/ringkasan.json", "w"), ensure_ascii=False, indent=1)
print("\nSelesai. Tabel di output/tabel, grafik di output/gambar.")